# Controlled car preprocessing benchmark · frozen DINOv1

Run top to bottom with `preprocessing_benchmark.py` uploaded next to this notebook (or clone your repository and `%cd` into it). The original notebook is preserved. This phase fixes the backbone/head and changes only preprocessing. No benchmark results are bundled. See README for full data and cache contracts.

## 1. Dependencies and environment
Use a Colab T4 runtime. Keep its compatible preinstalled torch/torchvision pair. This installation cell does not upgrade them.

In [ ]:
%pip -q install timm==1.0.30 scikit-learn pandas pillow tqdm matplotlib


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
from pathlib import Path
import torch, torchvision, timm
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__, 'timm:', timm.__version__)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (slower)')
assert Path('preprocessing_benchmark.py').is_file(), 'Upload the helper beside this notebook or change directory to your repository'
import importlib
import preprocessing_benchmark as benchmark
benchmark = importlib.reload(benchmark)


## 2. Optional Google Drive
Uncomment to mount Drive. Outputs can persist there between runtimes. Do not save API credentials in this notebook.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')


## 3. Configuration and dataset
Select one ID here; no transform edits are needed elsewhere. Initial mode runs P0–P4 with seed 123. Repeat mode runs only your explicitly selected promising configurations. Expected split counts are checked before extraction.

In [ ]:
DATA_ROOT = Path('/content/data')
OUTPUT_ROOT = Path('/content/car_preprocessing_outputs')  # Or a mounted Drive folder.
EXPERIMENT_ID = 'P0'
GLOBAL_SEED = 123
AUGMENTATION_SEED = 123
HEAD_SEED = 123
EXPECTED_COUNTS = (700, 200, 700)
FORCE_REEXTRACT = False
MAKE_SUBMISSION = False
MODE = 'single'  # 'single', 'initial', or 'repeat'
PROMISING_IDS = ['P0']  # Choose after reviewing the initial benchmark.
SEEDS = [42, 123, 2026]
config = benchmark.Config(experiment_id=EXPERIMENT_ID, global_seed=GLOBAL_SEED,
    augmentation_seed=AUGMENTATION_SEED, head_seed=HEAD_SEED, expected_counts=EXPECTED_COUNTS)
print(config)


## 4. Preprocessing, frozen DINOv1, extraction and safe caching

The helper defines all reusable functions. P0 uses a single deterministic bicubic view; P1/P2 use one random view; P3/P4 use one deterministic plus four random views. P2/P4 add ColorJitter. Validation/test always remain deterministic, with no TTA. View-major concatenation repeats the full training label vector per view. Seed once per extraction, not per view.

`load_backbone` freezes DINOv1 ViT-S/16 and verifies 384-D features. `get_features` fingerprints the actual weights and configuration, hashes dataset content, validates cached metadata/labels/shapes and honors forced regeneration. Extraction uses `inference_mode`, `eval`, sequential loading and zero workers. Changing only HEAD_SEED reuses the feature cache.

## 5. Standardization, linear training and validation

`standardize` uses training views only. `train_head` creates a fresh Linear(384,20) and AdamW for each run; defaults stay 35 epochs, lr=.001, weight decay=.001, patience=7, batch size=64. Validation accuracy selects the best epoch; macro-F1 and confusion matrices use that same checkpoint. Test labels are never read.

The following cell performs the selected workflow. It can download pretrained weights once. Real data is required; an absent dataset fails before any training.

In [ ]:
if MODE == 'single':
    record = benchmark.run_experiment(config, DATA_ROOT, OUTPUT_ROOT,
        force_reextract=FORCE_REEXTRACT, make_submission=MAKE_SUBMISSION)
    results, summary = benchmark.compare_results([record], OUTPUT_ROOT)
elif MODE in ('initial', 'repeat'):
    ids = list(benchmark.SPECS) if MODE == 'initial' else PROMISING_IDS
    seeds = [123] if MODE == 'initial' else SEEDS
    # Explicit counts supported here too; backbone loaded once for the sequence.
    benchmark.set_seed(GLOBAL_SEED)
    model = benchmark.load_backbone(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
    records = []
    for experiment_id in ids:
        for seed in seeds:
            run_config = benchmark.Config(experiment_id=experiment_id, global_seed=GLOBAL_SEED,
                augmentation_seed=seed, head_seed=seed, expected_counts=EXPECTED_COUNTS)
            records.append(benchmark.run_experiment(run_config, DATA_ROOT, OUTPUT_ROOT,
                FORCE_REEXTRACT, MAKE_SUBMISSION, backbone=model))
    results, summary = benchmark.compare_results(records, OUTPUT_ROOT)
else:
    raise ValueError('Unknown MODE')


## 6. Comparison, plots and exports
These are actual completed runs only. SD is sample SD and is undefined for one run. Select a small promising subset using accuracy, secondary macro-F1 and seed variability. With 200 validation images one correct prediction changes accuracy by 0.5 percentage points. Multi-view embeddings are correlated and also increase optimizer steps per epoch. Do not tune against Kaggle test results.

In [ ]:
display(results[['experiment_id','augmentation_seed','head_seed','train_embeddings','best_epoch','val_accuracy','macro_f1','cache_id']])
display(summary)
print('Artifacts:', OUTPUT_ROOT.resolve())
from IPython.display import Image, display
comparison_id = benchmark.digest(sorted(results.run_id.tolist()))[:12]
plot_names = [f'{run_id}_history.png' for run_id in results.run_id]
plot_names += [f'comparison_{comparison_id}.png', f'seed_summary_{comparison_id}.png']
for plot in [OUTPUT_ROOT / 'plots' / name for name in plot_names if (OUTPUT_ROOT / 'plots' / name).exists()]:
    print(plot.name)
    display(Image(filename=str(plot)))


## 7. Optional Kaggle submission
Set MAKE_SUBMISSION=True and rerun the execution cell to write a separate seed/configuration CSV under submissions/. Cached features are reused, but the fresh classifier retrains deterministically. CSV IDs follow sample_submission.csv (or sorted stems when absent). No test labels or random test augmentation are used. Validation accuracy is not leaderboard accuracy. Full GPU experiments and private data validation remain for your Colab run.